# SAE Interpretability Explorer

Interactive analysis + visualization of a trained Sparse Autoencoder on the frozen MGN node embeddings (per arXiv:2507.16069).

Loads an **existing** run (produced by `train_mgn` -> `extract_embeddings` -> `train_sae`), then lets you:
- Rank Top-K salient latents by the three Table-1 scores (Variance, MeanAbs, Entropy).
- Inspect any latent's activation histogram.
- Visualize a latent's **spatial activation** on the most-activated frame.

All plotting goes through `src/visualize.py` (shared with the CLI).

In [1]:
# --- Setup ---
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)

from src.env import get_env
from src.visualize import top_latents_bar, latent_histogram, spatial_scatter
from scripts.analyze import load_all_codes, salient_scores
from src.data.cylinder_flow import split_reader, build_sample
import ipywidgets as widgets
from IPython.display import display
print('setup OK; repo:', REPO_ROOT)

setup OK; repo: /Users/pjpekala/projects/cfd_interpretability


In [2]:
# --- Load a run ---
# Pick the run_name whose SAE/embeddings you want to explore, and the split.
run_name_w = widgets.Text(value='norm-smoke', description='run_name')
split_w = widgets.Dropdown(options=['train', 'valid', 'test'], value='test', description='split')
display(run_name_w, split_w)

env = get_env(hardware='auto', run_name=run_name_w.value, stage='analysis', seed=0)
split = split_w.value

# Codes (z) for every node-vector, plus the saved normalization stats.
codes, _ = load_all_codes(env.embed_dir, split, env.sae_ckpt_dir)
M, hidden = codes.shape
print(f'split={split}  samples={M}  hidden={hidden}')

# Paper Table-1 saliency scores.
scores = salient_scores(codes, bins=50)
for name, s in scores.items():
    print(f'  {name}: max={s.max():.4f}  mean={s.mean():.4f}')

TOP_K = 20
top = {name: np.argsort(-s)[:TOP_K] for name, s in scores.items()}

Text(value='norm-smoke', description='run_name')

Dropdown(description='split', index=2, options=('train', 'valid', 'test'), value='test')

split=test  samples=1151877  hidden=512


  variance: max=0.8226  mean=0.1131
  mean_abs: max=0.4795  mean=0.2053
  entropy: max=3.4359  mean=2.1793


In [3]:
# --- Top-K salient latents (Table 1) ---
for metric in ['variance', 'mean_abs', 'entropy']:
    fig = top_latents_bar(scores, top[metric], title=f'Top-{TOP_K} by {metric}')
    display(fig)

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

## Interactive latent inspection

Slide to choose a latent index. The histogram shows its activation distribution across all samples; below it, the **most-activated frame** is shown as a spatial heatmap (node color = that latent's activation).

In [4]:
# --- Per-latent interactivity ---
def frame_positions(data_dir, split, ex, fr):
    """Return (N,2) mesh positions for one (example, frame)."""
    example = next((e for i, e in enumerate(split_reader(data_dir, split)) if i == ex))
    return example['mesh_pos']  # [N, 2]

def top_frame_for_latent(latent_idx):
    """Find the single (example, frame) whose mean activation of `latent_idx` is highest."""
    # Load the SAE once (reused across files).
    import torch
    from src.models.sae import SAEConfig, SparseAutoencoder
    embed_dim = int(np.load(sorted((env.embed_dir / split).glob('*.npy'))[0]).shape[1])
    ck = __import__('src.utils.checkpoint', fromlist=['load_latest']).load_latest(env.sae_ckpt_dir)
    sae = SparseAutoencoder(SAEConfig(input_dim=embed_dim))
    sae.load_state_dict(ck['model_state'])
    sae.eval()
    stats = __import__('scripts.analyze', fromlist=['load_normalizer']).load_normalizer(env.embed_dir)
    best_ex, best_fr, best_mean = -1, -1, -1.0
    for p in sorted((env.embed_dir / split).glob('ex*_fr*.npy')):
        stem = p.stem  # ex{ex}_fr{fr}
        ex = int(stem.split('_')[0][2:])
        fr = int(stem.split('_')[1][2:])
        arr = np.load(p).astype('float32')
        if stats is not None:
            mean, std = stats
            arr = (arr - mean) / std
        with torch.no_grad():
            zc = sae.encode(torch.as_tensor(arr)).numpy()
        mean_act = float(zc[:, latent_idx].mean())
        if mean_act > best_mean:
            best_mean, best_ex, best_fr = mean_act, ex, fr
    return best_ex, best_fr

latent_slider = widgets.IntSlider(value=0, min=0, max=hidden - 1, step=1, description='latent')
display(latent_slider)

# Cache the SAE + normalizer so slider moves stay snappy.
import torch
from src.models.sae import SAEConfig, SparseAutoencoder
embed_dim = int(np.load(sorted((env.embed_dir / split).glob('*.npy'))[0]).shape[1])
_ck = __import__('src.utils.checkpoint', fromlist=['load_latest']).load_latest(env.sae_ckpt_dir)
_sae = SparseAutoencoder(SAEConfig(input_dim=embed_dim)); _sae.load_state_dict(_ck['model_state']); _sae.eval()
_stats = __import__('scripts.analyze', fromlist=['load_normalizer']).load_normalizer(env.embed_dir)
def _codes_for(arr):
    if _stats is not None:
        mean, std = _stats; arr = (arr - mean) / std
    with torch.no_grad():
        return _sae.encode(torch.as_tensor(arr.astype('float32'))).numpy()

def inspect(latent_idx):
    display(latent_histogram(codes, latent_idx))
    ex, fr = top_frame_for_latent(latent_idx)
    pos = frame_positions(env.data_dir, split, ex, fr)
    p = sorted((env.embed_dir / split).glob(f'ex{ex:05d}_fr{fr:04d}.npy'))[0]
    node_act = _codes_for(np.load(p))[:, latent_idx]
    fig = spatial_scatter(pos, node_act, title=f'latent {latent_idx}: top frame ex{ex} fr{fr}')
    display(fig)
    print(f'most-activated frame: ex={ex} fr={fr}  mean activation={node_act.mean():.4f}')

out = widgets.interactive_output(inspect, {'latent_idx': latent_slider})

IntSlider(value=0, description='latent', max=511)

In [5]:
# Render the interactive inspector (slider + plots).
display(latent_slider, out)

IntSlider(value=0, description='latent', max=511)

Output(outputs=({'output_type': 'display_data', 'metadata': {}, 'data': {'text/plain': '<Figure size 700x400 w…